# Dataset Inspection and Statistics

In [1]:
# ============================================================
# Section 1 — Setup, Dataset Loading, and Reproducibility
# ============================================================
!pip install -q datasets seqeval sklearn-crfsuite transformers accelerate evaluate

import random
import numpy as np
import torch
import pandas as pd

from pprint import pprint
from collections import Counter
from datasets import load_dataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Random seed fixed as:", SEED)

# ------------------------------------------------------------
# Load CoNLL-2003 dataset
# ------------------------------------------------------------
dataset = load_dataset(
    "eriktks/conll2003",
    revision="convert/parquet"
)

print("\nDataset loaded successfully:")
print(dataset)

# ------------------------------------------------------------
# Inspect one example
# ------------------------------------------------------------

example = dataset["train"][0]

print("\nFirst training example:")
pprint(example)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.7 MB/s eta 0:00:00
Random seed fixed as: 42


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


Dataset loaded successfully:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

First training example:
{'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'id': '0',
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'tokens': ['EU',
            'rejects',
            'German',
            'call',
            'to',
            'boycott',
            'British',
            'lamb',
            '.']}


In [2]:
# ============================================================
# Section 1 — BIO Labels and Dataset Statistics
# ============================================================

# ------------------------------------------------------------
# Extract NER label names
# ------------------------------------------------------------

label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print("Number of NER labels:", num_labels)
print("NER label names:")
print(label_names)

# ------------------------------------------------------------
# Print first sentence with BIO labels
# ------------------------------------------------------------

example = dataset["train"][0]

print("\nFirst sentence with readable BIO labels:")
for token, tag_id in zip(example["tokens"], example["ner_tags"]):
    print(f"{token:15s} {label_names[tag_id]}")

# ------------------------------------------------------------
# Dataset statistics
# ------------------------------------------------------------

def get_dataset_stats(split_name):
    num_sentences = len(dataset[split_name])
    num_tokens = sum(len(example["tokens"]) for example in dataset[split_name])

    label_counter = Counter()
    for example in dataset[split_name]:
        for tag_id in example["ner_tags"]:
            label_counter[label_names[tag_id]] += 1

    return num_sentences, num_tokens, label_counter


stats = {}

for split_name in ["train", "validation", "test"]:
    num_sentences, num_tokens, label_counter = get_dataset_stats(split_name)
    stats[split_name] = {
        "sentences": num_sentences,
        "tokens": num_tokens,
        "labels": label_counter
    }

split_stats_df = pd.DataFrame([
    {
        "Split": split_name,
        "Sentences": stats[split_name]["sentences"],
        "Tokens": stats[split_name]["tokens"]
    }
    for split_name in ["train", "validation", "test"]
])

label_freq_df = pd.DataFrame(
    stats["train"]["labels"].items(),
    columns=["Label", "Train Count"]
).sort_values("Train Count", ascending=False)

print("\nDataset split statistics:")
display(split_stats_df)

print("\nTraining label frequencies:")
display(label_freq_df)

# Save for later report writing
split_stats_df.to_csv("q2_dataset_split_statistics.csv", index=False)
label_freq_df.to_csv("q2_train_label_frequencies.csv", index=False)

print("\nSaved CSV files:")
print("- q2_dataset_split_statistics.csv")
print("- q2_train_label_frequencies.csv")

Number of NER labels: 9
NER label names:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

First sentence with readable BIO labels:
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O

Dataset split statistics:


,Split,Sentences,Tokens
0,train,14041,203621
1,validation,3250,51362
2,test,3453,46435



Training label frequencies:


,Label,Train Count
1,O,169578
5,B-LOC,7140
3,B-PER,6600
0,B-ORG,6321
4,I-PER,4528
6,I-ORG,3704
2,B-MISC,3438
8,I-LOC,1157
7,I-MISC,1155



Saved CSV files:
- q2_dataset_split_statistics.csv
- q2_train_label_frequencies.csv


# CRF Baseline Model

In [3]:
# ============================================================
# Section 2 — CRF Baseline: Feature Extraction and Data Preparation
# ============================================================

from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------
# seqeval evaluates entity-level precision, recall, and F1.
# This is more appropriate than plain token accuracy for NER.

def evaluate_ner(true_labels, predicted_labels, model_name="Model"):
    precision = precision_score(true_labels, predicted_labels)
    recall = recall_score(true_labels, predicted_labels)
    f1 = f1_score(true_labels, predicted_labels)

    print(f"{model_name} Results")
    print("=" * 60)
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print()
    print(classification_report(true_labels, predicted_labels))

    return {
        "model": model_name,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

# ------------------------------------------------------------
# CRF feature extraction
# ------------------------------------------------------------
# The CRF works on original word-level tokens.
# We manually define local token features and neighboring-token features.

def word2features(sentence_tokens, index):
    word = sentence_tokens[index]

    features = {
        "bias": 1.0,

        # Current word identity and shape features
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:2]": word[:2],
        "word[:3]": word[:3],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),

        # Simple orthographic features
        "contains-hyphen": "-" in word,
        "contains-dot": "." in word,
        "is-alpha": word.isalpha(),
    }

    # Previous token features
    if index > 0:
        prev_word = sentence_tokens[index - 1]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
            "-1:word.isdigit()": prev_word.isdigit(),
        })
    else:
        features["BOS"] = True  # Beginning of sentence

    # Next token features
    if index < len(sentence_tokens) - 1:
        next_word = sentence_tokens[index + 1]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
            "+1:word.isdigit()": next_word.isdigit(),
        })
    else:
        features["EOS"] = True  # End of sentence

    return features


def sent2features(sentence_tokens):
    return [word2features(sentence_tokens, i) for i in range(len(sentence_tokens))]


def sent2labels(tag_ids):
    return [label_names[tag_id] for tag_id in tag_ids]


def build_crf_data(split_name):
    X = []
    y = []

    for example in dataset[split_name]:
        X.append(sent2features(example["tokens"]))
        y.append(sent2labels(example["ner_tags"]))

    return X, y

# ------------------------------------------------------------
# Build CRF train, validation, and test data
# ------------------------------------------------------------

X_train_crf, y_train_crf = build_crf_data("train")
X_val_crf, y_val_crf = build_crf_data("validation")
X_test_crf, y_test_crf = build_crf_data("test")

print("CRF data prepared successfully.")
print("Number of training sentences:", len(X_train_crf))
print("Number of validation sentences:", len(X_val_crf))
print("Number of test sentences:", len(X_test_crf))

print("\nExample token:")
print(dataset["train"][0]["tokens"][0])

print("\nExtracted CRF features for this token:")
pprint(X_train_crf[0][0])

print("\nGold BIO label for this token:")
print(y_train_crf[0][0])

CRF data prepared successfully.
Number of training sentences: 14041
Number of validation sentences: 3250
Number of test sentences: 3453

Example token:
EU

Extracted CRF features for this token:
{'+1:word.isdigit()': False,
 '+1:word.istitle()': False,
 '+1:word.isupper()': False,
 '+1:word.lower()': 'rejects',
 'BOS': True,
 'bias': 1.0,
 'contains-dot': False,
 'contains-hyphen': False,
 'is-alpha': True,
 'word.isdigit()': False,
 'word.istitle()': False,
 'word.isupper()': True,
 'word.lower()': 'eu',
 'word[-2:]': 'EU',
 'word[-3:]': 'EU',
 'word[:2]': 'EU',
 'word[:3]': 'EU'}

Gold BIO label for this token:
B-ORG


# CRF Training and Evaluation

In [4]:
# ============================================================
# Section 3 — CRF Training and Evaluation
# ============================================================

import sklearn_crfsuite

# ------------------------------------------------------------
# Train CRF model
# ------------------------------------------------------------
# c1: L1 regularization coefficient
# c2: L2 regularization coefficient
# max_iterations: maximum number of optimization iterations
# all_possible_transitions=True allows the model to consider transitions
# even if they were not observed in the training data.

crf_model = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)

print("Training CRF model...")
crf_model.fit(X_train_crf, y_train_crf)
print("CRF training completed.")

# ------------------------------------------------------------
# Validation evaluation
# ------------------------------------------------------------

print("\nPredicting validation set...")
y_val_pred_crf = crf_model.predict(X_val_crf)

crf_val_results = evaluate_ner(
    y_val_crf,
    y_val_pred_crf,
    model_name="CRF Validation"
)

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------
# We evaluate the test set after validation evaluation.
# In the final report, validation results are used for comparison/development,
# while test results are the final held-out performance.

print("\nPredicting test set...")
y_test_pred_crf = crf_model.predict(X_test_crf)

crf_test_results = evaluate_ner(
    y_test_crf,
    y_test_pred_crf,
    model_name="CRF Test"
)

# ------------------------------------------------------------
# Store CRF results in a small table
# ------------------------------------------------------------

crf_results_df = pd.DataFrame([
    {
        "Model": "CRF",
        "Split": "Validation",
        "Precision": crf_val_results["precision"],
        "Recall": crf_val_results["recall"],
        "F1": crf_val_results["f1"]
    },
    {
        "Model": "CRF",
        "Split": "Test",
        "Precision": crf_test_results["precision"],
        "Recall": crf_test_results["recall"],
        "F1": crf_test_results["f1"]
    }
])

print("\nCRF summary results:")
display(crf_results_df)

crf_results_df.to_csv("q2_crf_results.csv", index=False)

print("\nSaved:")
print("- q2_crf_results.csv")

Training CRF model...
CRF training completed.

Predicting validation set...
CRF Validation Results
Precision: 0.8969
Recall:    0.8650
F1-score:  0.8807

              precision    recall  f1-score   support

         LOC       0.92      0.90      0.91      1837
        MISC       0.92      0.83      0.87       922
         ORG       0.85      0.81      0.83      1341
         PER       0.89      0.88      0.89      1842

   micro avg       0.90      0.87      0.88      5942
   macro avg       0.90      0.86      0.88      5942
weighted avg       0.90      0.87      0.88      5942


Predicting test set...
CRF Test Results
Precision: 0.8197
Recall:    0.7923
F1-score:  0.8058

              precision    recall  f1-score   support

         LOC       0.85      0.85      0.85      1668
        MISC       0.77      0.74      0.76       702
         ORG       0.80      0.72      0.76      1661
         PER       0.83      0.83      0.83      1617

   micro avg       0.82      0.79      0.81

,Model,Split,Precision,Recall,F1
0,CRF,Validation,0.896877,0.865029,0.880665
1,CRF,Test,0.819747,0.792316,0.805798



Saved:
- q2_crf_results.csv


# Transformer Model: DistilBERT Token Classification

In [5]:
# ============================================================
# Section 4 — Transformer Model: Tokenizer, Model, and Label Alignment
# ============================================================

from transformers import AutoTokenizer, AutoModelForTokenClassification

# ------------------------------------------------------------
# Model choice
# ------------------------------------------------------------
# We use distilbert-base-cased as a BERT-like transformer model.
# "cased" is important for NER because capitalization is informative.
# Example: "Turkey" as a location vs "turkey" as an animal/food.

model_checkpoint = "distilbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

bert_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("Tokenizer and transformer model loaded successfully.")
print("Model checkpoint:", model_checkpoint)
print("Number of labels:", num_labels)

# ------------------------------------------------------------
# Token-label alignment
# ------------------------------------------------------------
# CoNLL-2003 gives labels at word level.
# DistilBERT uses subword tokenization.
#
# Example:
# Original word:      Washingtonian
# Possible subwords:  Washington ##ian
#
# The dataset has one label for the original word, but the tokenizer may
# produce multiple subword pieces. Therefore, we align labels as follows:
#
# - Special tokens such as [CLS] and [SEP] get -100.
# - The first subword of each original word receives the original word label.
# - Later subwords receive -100 and are ignored during loss computation.
#
# PyTorch's CrossEntropyLoss ignores label id -100 by default.

MAX_LENGTH = 128

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    aligned_labels = []

    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                # Special tokens: [CLS], [SEP], padding
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # First subword of a word
                label_ids.append(labels[word_id])
            else:
                # Other subwords of the same word
                label_ids.append(-100)

            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

# ------------------------------------------------------------
# Apply tokenization and alignment to all splits
# ------------------------------------------------------------

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("\nTokenized dataset:")
print(tokenized_dataset)

# ------------------------------------------------------------
# Inspect one aligned example
# ------------------------------------------------------------

sample_index = 0

original_tokens = dataset["train"][sample_index]["tokens"]
original_labels = [label_names[tag_id] for tag_id in dataset["train"][sample_index]["ner_tags"]]

tokenized_example = tokenized_dataset["train"][sample_index]
tokens_after_tokenization = tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"])
aligned_label_ids = tokenized_example["labels"]

print("\nOriginal word-level tokens and labels:")
for tok, lab in zip(original_tokens, original_labels):
    print(f"{tok:15s} {lab}")

print("\nAfter DistilBERT subword tokenization and label alignment:")
for tok, lab_id in zip(tokens_after_tokenization, aligned_label_ids):
    label_text = "IGNORED" if lab_id == -100 else label_names[lab_id]
    print(f"{tok:15s} {label_text}")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

...

Tokenizer and transformer model loaded successfully.
Model checkpoint: distilbert-base-cased
Number of labels: 9


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

Original word-level tokens and labels:
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O

After DistilBERT subword tokenization and label alignment:
[CLS]           IGNORED
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
la              O
##mb            IGNORED
.               O
[SEP]           IGNORED


# Transformer Training and Evaluation

In [6]:
# ============================================================
# Section 5 — Transformer Training and Evaluation
# Corrected version for newer Transformers Trainer API
# ============================================================

from transformers import DataCollatorForTokenClassification
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

# ------------------------------------------------------------
# Device check
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# ------------------------------------------------------------
# seqeval metric
# ------------------------------------------------------------

seqeval_metric = evaluate.load("seqeval")

def compute_metrics(prediction_output):
    logits, labels = prediction_output
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        current_true_labels = []
        current_true_predictions = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                current_true_labels.append(label_names[label_id])
                current_true_predictions.append(label_names[pred_id])

        true_labels.append(current_true_labels)
        true_predictions.append(current_true_predictions)

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01

print("Transformer hyperparameters:")
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)
print("Weight decay:", WEIGHT_DECAY)
print("Max sequence length:", MAX_LENGTH)

# ------------------------------------------------------------
# Training arguments
# ------------------------------------------------------------

try:
    training_args = TrainingArguments(
        output_dir="q2_distilbert_ner",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none"
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir="q2_distilbert_ner",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        seed=SEED,
        report_to="none"
    )

# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------
# Newer versions of Transformers replaced tokenizer=... with processing_class=...

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("\nTraining DistilBERT token classification model...")
trainer.train()
print("Transformer training completed.")

# ------------------------------------------------------------
# Validation evaluation
# ------------------------------------------------------------

print("\nEvaluating DistilBERT on validation set...")
bert_val_metrics = trainer.evaluate(tokenized_dataset["validation"])
print(bert_val_metrics)

# ------------------------------------------------------------
# Test evaluation
# ------------------------------------------------------------

print("\nEvaluating DistilBERT on test set...")
bert_test_metrics = trainer.evaluate(tokenized_dataset["test"])
print(bert_test_metrics)

Using device: cuda


Transformer hyperparameters:
Batch size: 16
Learning rate: 2e-05
Epochs: 3
Weight decay: 0.01
Max sequence length: 128

Training DistilBERT token classification model...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.061016,0.056706,0.903441,0.911051,0.907230,0.984276
2,0.025176,0.049357,0.923373,0.929751,0.926551,0.987686
3,0.020800,0.045914,0.930948,0.938005,0.934463,0.988913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Transformer training completed.

Evaluating DistilBERT on validation set...


{'eval_loss': 0.045914486050605774, 'eval_precision': 0.9309480020063534, 'eval_recall': 0.9380053908355795, 'eval_f1': 0.9344633716539397, 'eval_accuracy': 0.9889133526878787, 'eval_runtime': 2.6816, 'eval_samples_per_second': 1211.95, 'eval_steps_per_second': 76.073, 'epoch': 3.0}

Evaluating DistilBERT on test set...
{'eval_loss': 0.11956265568733215, 'eval_precision': 0.8805064169268123, 'eval_recall': 0.8995393338058115, 'eval_f1': 0.8899211218229622, 'eval_accuracy': 0.9785638882306051, 'eval_runtime': 2.8476, 'eval_samples_per_second': 1212.589, 'eval_steps_per_second': 75.853, 'epoch': 3.0}


# Results Comparison

In [7]:
# ============================================================
# Section 6 — Results Comparison
# ============================================================

results_df = pd.DataFrame([
    {
        "Model": "CRF",
        "Split": "Validation",
        "Precision": crf_val_results["precision"],
        "Recall": crf_val_results["recall"],
        "F1": crf_val_results["f1"]
    },
    {
        "Model": "CRF",
        "Split": "Test",
        "Precision": crf_test_results["precision"],
        "Recall": crf_test_results["recall"],
        "F1": crf_test_results["f1"]
    },
    {
        "Model": "DistilBERT",
        "Split": "Validation",
        "Precision": bert_val_metrics["eval_precision"],
        "Recall": bert_val_metrics["eval_recall"],
        "F1": bert_val_metrics["eval_f1"]
    },
    {
        "Model": "DistilBERT",
        "Split": "Test",
        "Precision": bert_test_metrics["eval_precision"],
        "Recall": bert_test_metrics["eval_recall"],
        "F1": bert_test_metrics["eval_f1"]
    }
])

# Round only for display. Keep original numeric values in results_df.
results_display_df = results_df.copy()
for col in ["Precision", "Recall", "F1"]:
    results_display_df[col] = results_display_df[col].round(4)

print("NER model comparison:")
display(results_display_df)

results_df.to_csv("q2_ner_results.csv", index=False)

print("\nSaved:")
print("- q2_ner_results.csv")

NER model comparison:


,Model,Split,Precision,Recall,F1
0,CRF,Validation,0.8969,0.8650,0.8807
1,CRF,Test,0.8197,0.7923,0.8058
2,DistilBERT,Validation,0.9309,0.9380,0.9345
3,DistilBERT,Test,0.8805,0.8995,0.8899



Saved:
- q2_ner_results.csv


# Error Analysis

In [8]:
# ============================================================
# Section 7 — Error Analysis
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Get DistilBERT predictions in word-level BIO format
# ------------------------------------------------------------

def get_transformer_predictions(split_name):
    predictions_output = trainer.predict(tokenized_dataset[split_name])

    logits = predictions_output.predictions
    labels = predictions_output.label_ids
    predictions = np.argmax(logits, axis=-1)

    true_labels = []
    true_predictions = []

    for prediction, label in zip(predictions, labels):
        current_true_labels = []
        current_true_predictions = []

        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:
                current_true_labels.append(label_names[label_id])
                current_true_predictions.append(label_names[pred_id])

        true_labels.append(current_true_labels)
        true_predictions.append(current_true_predictions)

    return true_labels, true_predictions


print("Getting DistilBERT validation predictions...")
y_val_true_bert, y_val_pred_bert = get_transformer_predictions("validation")

print("Getting DistilBERT test predictions...")
y_test_true_bert, y_test_pred_bert = get_transformer_predictions("test")

print("Prediction extraction completed.")

# ------------------------------------------------------------
# BIO entity span extraction
# ------------------------------------------------------------
# Converts BIO labels into entity spans:
# Example:
# ["B-LOC", "I-LOC", "O"] -> [(0, 1, "LOC")]

def extract_entities_from_bio(labels):
    entities = []
    start = None
    entity_type = None

    for i, label in enumerate(labels):
        if label == "O":
            if entity_type is not None:
                entities.append((start, i - 1, entity_type))
                start = None
                entity_type = None

        elif label.startswith("B-"):
            if entity_type is not None:
                entities.append((start, i - 1, entity_type))

            start = i
            entity_type = label[2:]

        elif label.startswith("I-"):
            current_type = label[2:]

            if entity_type is None:
                # Invalid BIO sequence recovery:
                # Treat I-X without previous entity as beginning of X.
                start = i
                entity_type = current_type

            elif current_type != entity_type:
                # Entity type changed inside an I-tag sequence.
                # Close previous entity and start a new one.
                entities.append((start, i - 1, entity_type))
                start = i
                entity_type = current_type

    if entity_type is not None:
        entities.append((start, len(labels) - 1, entity_type))

    return entities

# ------------------------------------------------------------
# Error categorization for one sentence
# ------------------------------------------------------------

def analyze_sentence_errors(tokens, true_labels, pred_labels):
    gold_entities = extract_entities_from_bio(true_labels)
    pred_entities = extract_entities_from_bio(pred_labels)

    gold_set = set(gold_entities)
    pred_set = set(pred_entities)

    missed = gold_set - pred_set
    extra = pred_set - gold_set

    errors = []

    for gold in missed:
        gold_start, gold_end, gold_type = gold
        gold_text = " ".join(tokens[gold_start:gold_end + 1])

        # Same span but wrong type
        same_span_wrong_type = [
            pred for pred in extra
            if pred[0] == gold_start and pred[1] == gold_end and pred[2] != gold_type
        ]

        # Overlapping span, maybe same or different type
        overlapping_preds = [
            pred for pred in extra
            if not (pred[1] < gold_start or pred[0] > gold_end)
        ]

        if same_span_wrong_type:
            pred = same_span_wrong_type[0]
            pred_text = " ".join(tokens[pred[0]:pred[1] + 1])

            errors.append({
                "error_type": "Entity type confusion",
                "gold_entity": gold_text,
                "gold_type": gold_type,
                "pred_entity": pred_text,
                "pred_type": pred[2]
            })

        elif overlapping_preds:
            pred = overlapping_preds[0]
            pred_text = " ".join(tokens[pred[0]:pred[1] + 1])

            errors.append({
                "error_type": "Boundary error",
                "gold_entity": gold_text,
                "gold_type": gold_type,
                "pred_entity": pred_text,
                "pred_type": pred[2]
            })

        else:
            errors.append({
                "error_type": "Missed entity",
                "gold_entity": gold_text,
                "gold_type": gold_type,
                "pred_entity": "",
                "pred_type": ""
            })

    # Spurious predictions: predicted entity does not overlap any gold entity
    for pred in extra:
        pred_start, pred_end, pred_type = pred
        pred_text = " ".join(tokens[pred_start:pred_end + 1])

        overlaps_gold = [
            gold for gold in gold_entities
            if not (gold[1] < pred_start or gold[0] > pred_end)
        ]

        if not overlaps_gold:
            errors.append({
                "error_type": "Spurious entity",
                "gold_entity": "",
                "gold_type": "",
                "pred_entity": pred_text,
                "pred_type": pred_type
            })

    return errors

# ------------------------------------------------------------
# Collect errors for a full split
# ------------------------------------------------------------

def collect_errors(split_name, true_labels_list, pred_labels_list):
    all_errors = []

    for i in range(len(dataset[split_name])):
        tokens = dataset[split_name][i]["tokens"]
        true_labels = true_labels_list[i]
        pred_labels = pred_labels_list[i]

        sentence_errors = analyze_sentence_errors(tokens, true_labels, pred_labels)

        for error in sentence_errors:
            error["sentence_id"] = i
            error["sentence"] = " ".join(tokens)
            all_errors.append(error)

    return pd.DataFrame(all_errors)

# ------------------------------------------------------------
# Validation error analysis
# ------------------------------------------------------------
# We use validation errors for qualitative analysis because the test set
# should mainly be reserved for final quantitative reporting.

crf_val_errors_df = collect_errors(
    "validation",
    y_val_crf,
    y_val_pred_crf
)

bert_val_errors_df = collect_errors(
    "validation",
    y_val_true_bert,
    y_val_pred_bert
)

print("\nCRF validation error type counts:")
crf_error_counts = crf_val_errors_df["error_type"].value_counts()
display(crf_error_counts)

print("\nDistilBERT validation error type counts:")
bert_error_counts = bert_val_errors_df["error_type"].value_counts()
display(bert_error_counts)

print("\nSample CRF validation errors:")
display(crf_val_errors_df.head(10))

print("\nSample DistilBERT validation errors:")
display(bert_val_errors_df.head(10))

# Save error analysis files
crf_val_errors_df.to_csv("q2_crf_validation_errors.csv", index=False)
bert_val_errors_df.to_csv("q2_distilbert_validation_errors.csv", index=False)

print("\nSaved:")
print("- q2_crf_validation_errors.csv")
print("- q2_distilbert_validation_errors.csv")

Getting DistilBERT validation predictions...


Getting DistilBERT test predictions...


Prediction extraction completed.

CRF validation error type counts:


,count
error_type,
Entity type confusion,364
Missed entity,264
Boundary error,174
Spurious entity,50



DistilBERT validation error type counts:


...


Sample CRF validation errors:


,error_type,gold_entity,gold_type,pred_entity,pred_type,sentence_id,sentence
0,Entity type confusion,West Indian,MISC,West Indian,LOC,2,West Indian all-rounder Phil Simmons took four...
1,Missed entity,Simmons,PER,,,5,"Trailing by 213 , Somerset got a solid start t..."
2,Entity type confusion,Hussain,PER,Hussain,LOC,7,"Hussain , considered surplus to England 's one..."
3,Missed entity,Such,PER,,,8,By the close Yorkshire had turned that into a ...
4,Missed entity,ex-England,MISC,,,14,They were held up by a gritty 84 from Paul Joh...
5,Missed entity,Chester-le-Street,LOC,,,22,Chester-le-Street : Glamorgan 259 and 207 ( A....
6,Missed entity,T.O'Gorman,PER,,,27,"Chesterfield : Worcestershire 238 and 133-5 , ..."
7,Missed entity,ASHES,MISC,,,29,CRICKET - 1997 ASHES INTINERARY .
8,Missed entity,Ashes,MISC,,,31,Australia will defend the Ashes in
9,Spurious entity,,,Tour,MISC,41,Tour itinerary :



Sample DistilBERT validation errors:


,error_type,gold_entity,gold_type,pred_entity,pred_type,sentence_id,sentence
0,Entity type confusion,LEICESTERSHIRE,ORG,LEICESTERSHIRE,MISC,0,CRICKET - LEICESTERSHIRE TAKE OVER AT TOP AFTE...
1,Missed entity,ex-England,MISC,,,14,They were held up by a gritty 84 from Paul Joh...
2,Missed entity,ASHES,MISC,,,29,CRICKET - 1997 ASHES INTINERARY .
3,Boundary error,Minor Counties XI,ORG,Minor Counties,ORG,66,July 9 v Minor Counties XI
4,Spurious entity,,,INTERNATIONAL,MISC,88,BASKETBALL - INTERNATIONAL TOURNAMENT RESULT .
5,Spurious entity,,,UNDER-21,LOC,94,SOCCER - ROMANIA BEAT LITHUANIA IN UNDER-21 MA...
6,Missed entity,ROTOR,ORG,,,101,SOCCER - ROTOR FANS LOCKED OUT AFTER VOLGOGRAD...
7,Missed entity,VOLGOGRAD,LOC,,,101,SOCCER - ROTOR FANS LOCKED OUT AFTER VOLGOGRAD...
8,Entity type confusion,Rotor,ORG,Rotor,PER,103,Rotor Volgograd must play their next home game...
9,Entity type confusion,Rotor,ORG,Rotor,PER,104,The head of the Russian league 's disciplinary...



Saved:
- q2_crf_validation_errors.csv
- q2_distilbert_validation_errors.csv


In [9]:
def print_error_examples(error_df, model_name, error_type, n=3):
    subset = error_df[error_df["error_type"] == error_type].head(n)

    print(f"\n{model_name}: {error_type}")
    print("=" * 100)

    if len(subset) == 0:
        print("No examples found for this error type.")
        return

    for _, row in subset.iterrows():
        print("Sentence ID:", row["sentence_id"])
        print("Sentence:", row["sentence"])
        print("Gold entity:", row["gold_entity"], f"({row['gold_type']})")
        print("Pred entity:", row["pred_entity"], f"({row['pred_type']})")
        print("-" * 100)


error_types = [
    "Boundary error",
    "Entity type confusion",
    "Missed entity",
    "Spurious entity"
]

print("CRF readable validation error examples")
print("#" * 100)

for error_type in error_types:
    print_error_examples(crf_val_errors_df, "CRF", error_type, n=3)


print("\n\nDistilBERT readable validation error examples")
print("#" * 100)

for error_type in error_types:
    print_error_examples(bert_val_errors_df, "DistilBERT", error_type, n=3)

CRF readable validation error examples
####################################################################################################

CRF: Boundary error
Sentence ID: 44
Sentence: May 14 Practice at Lord 's
Gold entity: Lord 's (LOC)
Pred entity: Lord (LOC)
----------------------------------------------------------------------------------------------------
Sentence ID: 53
Sentence: May 25 Third one-day international ( at Lord 's , London )
Gold entity: Lord 's (LOC)
Pred entity: Lord (LOC)
----------------------------------------------------------------------------------------------------
Sentence ID: 61
Sentence: June 19-23 Second test ( at Lord 's )
Gold entity: Lord 's (LOC)
Pred entity: Lord (LOC)
----------------------------------------------------------------------------------------------------

CRF: Entity type confusion
Sentence ID: 2
Sentence: West Indian all-rounder Phil Simmons took four for 38 on Friday as Leicestershire beat Somerset by an innings and 39 runs in two

In [10]:
# Save Final Tables
split_stats_df.to_csv("q2_dataset_split_statistics.csv", index=False)
label_freq_df.to_csv("q2_train_label_frequencies.csv", index=False)
results_df.to_csv("q2_ner_results.csv", index=False)
crf_val_errors_df.to_csv("q2_crf_validation_errors.csv", index=False)
bert_val_errors_df.to_csv("q2_distilbert_validation_errors.csv", index=False)

print("Saved final Question 2 tables:")
print("- q2_dataset_split_statistics.csv")
print("- q2_train_label_frequencies.csv")
print("- q2_ner_results.csv")
print("- q2_crf_validation_errors.csv")
print("- q2_distilbert_validation_errors.csv")

print("\nQuestion 2 pipeline completed successfully.")

Saved final Question 2 tables:
- q2_dataset_split_statistics.csv
- q2_train_label_frequencies.csv
- q2_ner_results.csv
- q2_crf_validation_errors.csv
- q2_distilbert_validation_errors.csv

Question 2 pipeline completed successfully.
